In [1]:
# ONLY RUN THIS IF YOU'RE IN GOOGLE COLAB
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Thesis/Pintu-Air')

# Verify you're in the right place
!pwd
!ls -la

Mounted at /content/drive
/content/drive/MyDrive/Thesis/Pintu-Air
total 24843
-rw------- 1 root root 5517847 Jun 11 14:58 '01 Result Data Cleaning Part 1.csv'
-rw------- 1 root root 1964974 Jun 11 14:58 '02 Data Preperation.ipynb'
-rw------- 1 root root  256098 Jun 11 14:58 '02 X_test.csv'
-rw------- 1 root root 4831129 Jun 11 14:58 '02 X_train.csv'
-rw------- 1 root root   40580 Jun 11 14:58 '02 y_test.csv'
-rw------- 1 root root  770651 Jun 11 14:58 '02 y_train.csv'
-rw------- 1 root root  322748 Jun 11 14:58 '03 ARIMA.ipynb'
-rw------- 1 root root    2277 Jun 11 14:58  03_Result_ARIMA.csv
-rw------- 1 root root 6247212 Jun 11 14:58  04c_ARIMA_Manggarai.ipynb
-rw------- 1 root root  631448 Jun 11 14:58  06_DataPreperation_ML.ipynb
-rw------- 1 root root 1611961 Jun 11 14:58  07a_best_model.keras
-rw------- 1 root root  371715 Jun 11 14:58  07a_LSTM_Manggarai.ipynb
-rw------- 1 root root 1685693 Jun 11 14:58  07b_best_model.keras
-rw------- 1 root root  368771 Jun 11 14:58  07b_LSTM_M

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Set style and random seed
plt.style.use('default')
np.random.seed(42)

In [3]:
X_train = pd.read_csv('05_X_train_binary.csv', index_col='Tanggal', parse_dates=True)
X_test = pd.read_csv('05_X_test_binary.csv', index_col='Tanggal', parse_dates=True)
y_train = pd.read_csv('04_y_train.csv', index_col='Tanggal', parse_dates=True).squeeze()
y_test = pd.read_csv('04_y_test.csv', index_col='Tanggal', parse_dates=True).squeeze()

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print(f"\nFeatures: {list(X_train.columns)}")


X_train shape: (33972, 151)
X_test shape: (1788, 151)

y_train shape: (33972,)
y_test shape: (1788,)

Features: ['time_index', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos', 'manggarai_air_lag1', 'manggarai_air_lag2', 'manggarai_air_lag3', 'manggarai_air_lag4', 'manggarai_air_lag5', 'manggarai_air_lag6', 'manggarai_air_lag7', 'manggarai_air_lag8', 'manggarai_air_lag9', 'manggarai_air_lag10', 'manggarai_air_lag11', 'manggarai_air_lag12', 'manggarai_air_lag13', 'manggarai_air_lag14', 'manggarai_air_lag15', 'manggarai_air_lag16', 'manggarai_air_lag17', 'manggarai_air_lag18', 'manggarai_air_lag19', 'manggarai_air_lag20', 'manggarai_air_lag21', 'manggarai_air_lag22', 'manggarai_air_lag23', 'manggarai_air_lag24', 'depok_air_lag1', 'depok_air_lag2', 'depok_air_lag3', 'depok_air_lag4', 'depok_air_lag5', 'depok_air_lag6', 'depok_air_lag7', 'depok_air_lag8', 'depok_air_lag9', 'depok_air_lag10', 'depok_air_lag11', 'depok_air_lag12', 'depok_air_lag13',

# Baseline Model

In [4]:
fixed_params = {
    'tree_method': 'gpu_hist',
    'gpu_id': 0,
    'objective': 'reg:squarederror',
    'n_jobs': -1,
    'random_state': 42,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

results = []


start_time = datetime.now()

# Train Baseline
model_base = xgb.XGBRegressor(**fixed_params)
model_base.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

training_time = datetime.now() - start_time
training_time_sec = training_time.total_seconds() 

# Hitung Metrik Baseline
base_train_pred = model_base.predict(X_train)
base_test_pred = model_base.predict(X_test)

b_tr_rmse = np.sqrt(mean_squared_error(y_train, base_train_pred))
b_tr_mae = mean_absolute_error(y_train, base_train_pred)
b_tr_mape = mean_absolute_percentage_error(y_train, base_train_pred)
b_tr_r2 = r2_score(y_train, base_train_pred)

b_te_rmse = np.sqrt(mean_squared_error(y_test, base_test_pred))
b_te_mae = mean_absolute_error(y_test, base_test_pred)
b_te_mape = mean_absolute_percentage_error(y_test, base_test_pred)
b_te_r2 = r2_score(y_test, base_test_pred)

# Simpan Baseline
results.append({
    'learning_rate': 'BASELINE',
    'n_estimators': 'BASELINE',
    'max_depth': 'BASELINE',
    
    # Masukkan Waktu Training
    'training_time_sec': round(training_time_sec, 4),
    
    'train_rmse': round(b_tr_rmse, 8),
    'train_mae': round(b_tr_mae, 8),
    'train_mape': round(b_tr_mape, 8),
    'train_r2': round(b_tr_r2, 8),
    'test_rmse': round(b_te_rmse, 8),
    'test_mae': round(b_te_mae, 8),
    'test_mape': round(b_te_mape, 8),
    'test_r2': round(b_te_r2, 8)
})

In [5]:
results

[{'learning_rate': 'BASELINE',
  'n_estimators': 'BASELINE',
  'max_depth': 'BASELINE',
  'training_time_sec': 3.5056,
  'train_rmse': np.float64(7.87083807),
  'train_mae': 2.84005279,
  'train_mape': 0.00705757,
  'train_r2': 0.96851561,
  'test_rmse': np.float64(24.69610177),
  'test_mae': 5.92856059,
  'test_mape': 0.01221772,
  'test_r2': -0.96099925}]

# Hyperparameter Tuning

In [6]:
# Parameter yang akan diubah-ubah (Grid)
param_learning_rate = [0.001, 0.005, 0.01, 0.03, 0.05, 0.1, 0.2]
param_n_estimators = [50, 100, 200, 500, 1000, 2000, 3000]
param_max_depth = [1, 2, 3, 4, 5, 6, 7, 8, 9]

In [7]:
len(param_learning_rate) * len(param_n_estimators) * len(param_max_depth)

441

In [8]:
total_combinations = len(param_learning_rate) * len(param_n_estimators) * len(param_max_depth)
counter = 0

for lr in param_learning_rate:
    for n_est in param_n_estimators:
        for depth in param_max_depth:
            counter += 1
            
            # Gabungkan parameter fixed dengan parameter loop saat ini
            current_params = fixed_params.copy()
            current_params['learning_rate'] = lr
            current_params['n_estimators'] = n_est
            current_params['max_depth'] = depth
            
            print(f"[{counter}/{total_combinations}] Training: lr={lr}, n_est={n_est}, depth={depth}")
            
            # --- Mulai Timer Loop ---
            loop_start_time = datetime.now()
            
            # Train Model
            model = xgb.XGBRegressor(**current_params)
            model.fit(
                X_train, y_train,
                eval_set=[(X_test, y_test)],
                verbose=False
            )
            
            # --- Stop Timer Loop ---
            loop_training_time = datetime.now() - loop_start_time
            loop_time_sec = loop_training_time.total_seconds()
            
            # Predict
            train_pred = model.predict(X_train)
            test_pred = model.predict(X_test)
            
            # Calculate Metrics (Train)
            tr_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
            tr_mae = mean_absolute_error(y_train, train_pred)
            tr_mape = mean_absolute_percentage_error(y_train, train_pred)
            tr_r2 = r2_score(y_train, train_pred)
            
            # Calculate Metrics (Test)
            te_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
            te_mae = mean_absolute_error(y_test, test_pred)
            te_mape = mean_absolute_percentage_error(y_test, test_pred)
            te_r2 = r2_score(y_test, test_pred)
            
            # Append to results
            results.append({
                'learning_rate': lr,
                'n_estimators': n_est,
                'max_depth': depth,
                
                # Masukkan Waktu Training
                'training_time_sec': round(loop_time_sec, 4),
                
                'train_rmse': round(tr_rmse, 8),
                'train_mae': round(tr_mae, 8),
                'train_mape': round(tr_mape, 8),
                'train_r2': round(tr_r2, 8),
                'test_rmse': round(te_rmse, 8),
                'test_mae': round(te_mae, 8),
                'test_mape': round(te_mape, 8),
                'test_r2': round(te_r2, 8)
            })

[1/441] Training: lr=0.001, n_est=50, depth=1
[2/441] Training: lr=0.001, n_est=50, depth=2
[3/441] Training: lr=0.001, n_est=50, depth=3
[4/441] Training: lr=0.001, n_est=50, depth=4
[5/441] Training: lr=0.001, n_est=50, depth=5
[6/441] Training: lr=0.001, n_est=50, depth=6
[7/441] Training: lr=0.001, n_est=50, depth=7
[8/441] Training: lr=0.001, n_est=50, depth=8
[9/441] Training: lr=0.001, n_est=50, depth=9
[10/441] Training: lr=0.001, n_est=100, depth=1
[11/441] Training: lr=0.001, n_est=100, depth=2
[12/441] Training: lr=0.001, n_est=100, depth=3
[13/441] Training: lr=0.001, n_est=100, depth=4
[14/441] Training: lr=0.001, n_est=100, depth=5
[15/441] Training: lr=0.001, n_est=100, depth=6
[16/441] Training: lr=0.001, n_est=100, depth=7
[17/441] Training: lr=0.001, n_est=100, depth=8
[18/441] Training: lr=0.001, n_est=100, depth=9
[19/441] Training: lr=0.001, n_est=200, depth=1
[20/441] Training: lr=0.001, n_est=200, depth=2
[21/441] Training: lr=0.001, n_est=200, depth=3
[22/441] T

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv('12_XGBoost_binary_GridSearch_Results_2.csv', index=False)